In [0]:
dim_customers = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_customers")
dim_invoices = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_invoices")
dim_products = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_products")
fact_invoice_line_items = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.fact_invoice_line_items")
fact_payments = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.fact_payments")

In [0]:
display(fact_invoice_line_items)

In [0]:
# Total Revenue – Sum of all invoice line-item amounts
from pyspark.sql.functions import sum as _sum

total_revenue = fact_invoice_line_items.agg(
    _sum("usd_revenue").alias("total_revenue")
)

display(total_revenue)

total_revenue.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_total_revenue")

In [0]:
#Total Quantity Sold – Sum of quantities sold across all products
from pyspark.sql.functions import sum as _sum

total_quantity_sold = fact_invoice_line_items.agg(
    _sum("quantity").alias("total_quantity_sold")
)

display(total_quantity_sold)

total_quantity_sold.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_total_quantity_sold")

In [0]:
# Average Invoice Value – Average total value per invoice
from pyspark.sql.functions import sum as _sum, avg

invoice_totals = fact_invoice_line_items.groupBy("invoice_id") \
    .agg(_sum("usd_revenue").alias("invoice_total"))

average_invoice_value = invoice_totals.agg(
    avg("invoice_total").alias("average_invoice_value")
)

display(average_invoice_value)

average_invoice_value.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_average_invoice_value")

In [0]:
# Top Customers by Revenue – Identify customers contributing the most revenue
from pyspark.sql.functions import sum as _sum

fact_with_customer = fact_invoice_line_items.join(
    dim_invoices.select("invoice_id", "customer_id"),
    "invoice_id"
)

customer_revenue = fact_with_customer.groupBy("customer_id") \
    .agg(_sum("usd_revenue").alias("total_revenue"))

customer_revenue = customer_revenue.join(dim_customers, "customer_id")

top_customers = customer_revenue.orderBy("total_revenue", ascending=False).limit(10)

display(top_customers)

top_customers.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_top_10_customers")

In [0]:
# Top Products by Revenue – Identify products generating the highest sales revenue
from pyspark.sql.functions import sum as _sum

product_revenue = fact_invoice_line_items.groupBy("product_id") \
    .agg(_sum("usd_revenue").alias("total_revenue"))

top_products = product_revenue.join(
    dim_products.dropDuplicates(["product_id"]),
    "product_id"
)

display(top_products)

top_products.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_product_revenue")

In [0]:
# Invoice Cancellation Rate – Percentage of invoices marked as "Cancelled" vs total invoices
from pyspark.sql.functions import col, sum as _sum, when

invoice_cancellation_df = dim_invoices.agg(
    (_sum(when(col("invoice_status") == "Cancelled", 1).otherwise(0)) / _sum(when(col("invoice_id").isNotNull(), 1).otherwise(0)) * 100)
    .alias("invoice_cancellation_rate")
)
display(invoice_cancellation_df)

invoice_cancellation_df.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_invoice_cancellation_rate")

In [0]:
from pyspark.sql.functions import sum as _sum, avg

fact_with_customer = fact_invoice_line_items.join(
    dim_invoices.select("invoice_id", "customer_id"),
    "invoice_id"
)

discount_metrics = fact_with_customer.groupBy("customer_id").agg(
    _sum("discount").alias("total_discount"),
    avg("discount").alias("avg_discount")
)

display(discount_metrics)

discount_metrics.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_discount_metrics")

In [0]:
from pyspark.sql.functions import sum as _sum

invoice_region = dim_invoices.select("invoice_id", "region")

revenue_by_region = fact_invoice_line_items.join(invoice_region, "invoice_id").groupBy("region").agg(_sum("usd_revenue").alias("total_revenue"))

display(revenue_by_region)

revenue_by_region.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_revenue_by_region")

In [0]:
# Number of Invoices per Customer – Measures customer purchase frequency
from pyspark.sql.functions import count

invoices_per_customer = dim_invoices.groupBy("customer_id") \
    .agg(count("invoice_id").alias("total_invoices"))

display(invoices_per_customer)

invoices_per_customer.write.mode("overwrite").saveAsTable("global_wholesale_distributor.reporting_wholesale_distributor.kpi_invoices_per_customer")